# 02: DAS pipeline on Monterey Bay earthquakes

Takes a DAS earthquake waterfall and turns every good/working channel into the
format the EQ/UNE classifiers expect (0.5-10 Hz, 20 Hz, 1200 samples,
P at sample 200, peak-normalized).

This is the DAS half of the transfer test. The models are trained on
seismograms; our question is whether they still work when the input is a
fibre-optic cable channel.

**Before running:** `python scripts/download_monterey.py --limit 3`

In [ ]:
import sys, os
sys.path.append("..") #so 'src' imports work from inside notebooks/

import numpy as np
import matplotlib.pyplot as plt

from src.das_io import load_das_event, select_channels, channel_snr
from src.preprocess import preprocess_trace, bandpass, pick_p
from src.features import extract_features, feature_table, RF_V2_FEATURES

## 1. Load one event

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/monterey_bay/data")
files = sorted(DATA_DIR.glob("*.h5"))
print(f"{len(files)} event files found")

data, meta = load_das_event(files[0])

print(f"\nevent{meta.get('event_id')}")
print(f"time{meta.get('event_time')}")
print(f"magnitude M{meta.get('magnitude')} {meta.get('magnitude_type', '')}")
print(f"location {meta.get('latitude')}, {meta.get('longitude')}  depth {meta.get('depth_km')} km")
print(f"shape {meta['n_channels']} channels x {meta['n_time']} samples")
print(f"rate {meta['fs']} Hz channel spacing {meta.get('dx_m')} m")
print(f"units {meta.get('unit')}")
print(f"duration {meta['n_time'] / meta['fs']:.0f} s")

## 2. Raw waterfall

Time down, distance along the cable across

In [ ]:
clip = np.percentile(np.abs(data), 98)

plt.figure(figsize=(12, 6))
plt.imshow(data.T, aspect="auto", cmap="seismic", vmin=-clip, vmax=clip,
           extent=[0, meta["n_channels"], meta["n_time"] / meta["fs"], 0])
plt.xlabel("Channel")
plt.ylabel("Time (s)")
plt.title(f"Monterey Bay DAS - {meta.get('event_id')} (M{meta.get('magnitude')})")
plt.colorbar(label=meta.get("unit", ""))
plt.tight_layout()
plt.show()

## 3. Picking the "good" channels

DAS gives thousands of channels and many are useless, like bad coupling to the
seafloor, local noise. Prof's handout's DAS test took 100 channels
per event chosen on SNR, so we tried doing the same.

`event_time_index` is the ORIGIN time, not the P arrival, so it is only a
lower bound for where the signal starts. We use it to split noise from signal
when scoring SNR.

In [ ]:
split= meta.get("event_time_index") or int(0.25 * meta["n_time"])
keep, snrs = select_channels(data, split_index=split, n=100, min_snr=2.0)

print(f"kept {len(keep)} of {meta['n_channels']} channels (SNR >= 2.0)")
if len(keep):
    print(f"SNR range: {snrs.min():.1f} to {snrs.max():.1f}")

plt.figure(figsize=(12, 3))
all_snr = np.array([channel_snr(tr, split) for tr in data])
plt.plot(all_snr, lw=0.8, color="0.4")
plt.axhline(2.0, color="crimson", ls="--", lw=1, label="SNR cutoff = 2.0")
plt.scatter(keep, snrs, s=8, color="crimson", zorder=3, label=f"kept ({len(keep)})")
plt.xlabel("Channel"); plt.ylabel("SNR"); plt.legend()
plt.title("Signal-to-noise ratio along the cable")
plt.tight_layout(); plt.show()

## 4. Before and after for one channel

Should show raw channel on top, processed 1200 sample
window below, with the P pick marked.

In [ ]:
ch = int(keep[0])
raw = data[ch]
window, info = preprocess_trace(raw, meta["fs"])

print(f"channel {ch}")
for k, v in info.items():
    print(f"  {k:20s} {v}")

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

t_raw = np.arange(len(raw)) / meta["fs"]
axes[0].plot(t_raw, raw, "k-", lw=0.4)
axes[0].axvline(info["p_time_s"], color="crimson", lw=1.5, label="P pick (STA/LTA)")
axes[0].set_title(f"Raw channel {ch} at {meta['fs']:.0f} Hz")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel(meta.get("unit", "")); axes[0].legend()

t_win = (np.arange(1200) - 200) / 20.0 # 0s = the P arrival
axes[1].plot(t_win, window, "k-", lw=0.6)
axes[1].axvline(0, color="crimson", lw=1.5, label="P (sample 200)")
axes[1].axvspan(-10, 0, color="0.85", label="pre-P noise")
axes[1].set_xlim(-10, 50)
axes[1].set_title("Processed: 0.5-10 Hz, 20 Hz, 1200 samples, peak-normalised")
axes[1].set_xlabel("Time relative to P (s)"); axes[1].set_ylabel("Normalised"); axes[1].legend()

plt.tight_layout(); plt.show()

## 5. Kept channels

Some channels will fail with no STA/LTA trigger, or from the 60s window running off
the end.

In [ ]:
windows, infos, used, failures = [], [], [], {}

for c in keep:
    try:
        w, i = preprocess_trace(data[int(c)], meta["fs"])
        windows.append(w); infos.append(i); used.append(int(c))
    except ValueError as e:
        reason=str(e).split("(")[0].strip(4)[:45]
        failures[reason] = failures.get(reason, 0) + 1

print(f"processed {len(windows)} of {len(keep)} channels")
for reason, n in failures.items():
    print(f"  skipped {n:3d}: {reason}")

### Moveout check

If the picks are real, they should drift smoothly along the cable as the
wavefront travels, not jump around at random.

In [ ]:
picks = np.array([i["p_time_s"] for i in infos])

plt.figure(figsize=(11, 4))
plt.scatter(used, picks, s=12, color="crimson")
plt.xlabel("Channel"); plt.ylabel("P pick time (s)")
plt.title("P pick vs channel - a smooth trend means the picks are tracking a real wavefront")
plt.tight_layout(); plt.show()

print(f"pick spread: {picks.min():.1f} to {picks.max():.1f} s  (std {picks.std():.2f} s)")

## 6. Features

The 13 RF-v2 features, one row per channel. This matrix is what gets fed
to the classifier.

In [ ]:
X, names = feature_table(windows)
print(f"feature matrix: {X.shape[0]} channels x {X.shape[1]} features")

import pandas as pd
df = pd.DataFrame(X, columns=names, index=used)
df.index.name = "channel"
display(df.describe().T[["mean", "std", "min", "max"]].round(3))

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 8))
for ax, name in zip(axes.ravel(), names):
    ax.hist(df[name], bins=25, color="steelblue", edgecolor="white", lw=0.4)
    ax.set_title(name, fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(names):]:
    ax.axis("off")
plt.suptitle(f"RF-v2 features across {len(df)} DAS channels - {meta.get('event_id')}", y=1.0)
plt.tight_layout(); plt.show()

## 7. Save

One CSV per event. Collect these across all 33 events and that is the DAS
earthquake corpus the classifier gets tested on.

In [ ]:
out = Path("../data/features")
out.mkdir(exist_ok=True, parents=True)

df["event_id"] = meta.get("event_id")
df["magnitude"] = meta.get("magnitude")
df["depth_km"] = meta.get("depth_km")
df["latitude"] = meta.get("latitude")
df["longitude"] = meta.get("longitude")
df["label"] = "earthquake"

path = out / f"{meta.get('event_id', files[0].stem)}_features.csv"
df.to_csv(path)
print(f"wrote {path}  ({len(df)} rows)")